In [1]:
import tkinter as tk
from tkinter import ttk, messagebox

In [2]:
class FCFSApp:
    def __init__(self, root):
        self.root = root
        self.root.title("FCFS Scheduling GUI")

        self.processes = []

        # ===== Input frame =====
        frame_input = tk.Frame(root)
        frame_input.pack(pady=10)

        tk.Label(frame_input, text="Process ID").grid(row=0, column=0)
        tk.Label(frame_input, text="Arrival Time").grid(row=0, column=1)
        tk.Label(frame_input, text="Burst Time").grid(row=0, column=2)

        self.pid_entry = tk.Entry(frame_input, width=10)
        self.at_entry = tk.Entry(frame_input, width=10)
        self.bt_entry = tk.Entry(frame_input, width=10)

        self.pid_entry.grid(row=1, column=0)
        self.at_entry.grid(row=1, column=1)
        self.bt_entry.grid(row=1, column=2)

        self.pid_entry.bind("<Return>", lambda e: self.at_entry.focus_set())
        self.at_entry.bind("<Return>", lambda e: self.bt_entry.focus_set())
        self.bt_entry.bind("<Return>", self.on_enter_add)

        tk.Button(frame_input, text="Add / Update", command=self.add_process).grid(row=1, column=3, padx=10)
        tk.Button(frame_input, text="Clear All", command=self.clear_all).grid(row=1, column=4, padx=5)
        self.delete_button = tk.Button(frame_input, text="Delete Selected", command=self.delete_selected_process, state=tk.DISABLED)
        self.delete_button.grid(row=1, column=5, padx=5)

        # ===== Table =====
        self.tree = ttk.Treeview(root, columns=("PID", "AT", "BT"), show="headings", height=5)
        self.tree.heading("PID", text="Process ID")
        self.tree.heading("AT", text="Arrival Time")
        self.tree.heading("BT", text="Burst Time")
        self.tree.pack(pady=10)
        self.tree.bind("<<TreeviewSelect>>", self.on_select_process)

        # ===== Run button =====
        tk.Button(root, text="Run FCFS", command=self.run_fcfs, font=("Arial", 10, "bold"), bg="#4CAF50", fg="white").pack(pady=5)

        # ===== Text Output =====
        self.output = tk.Text(root, height=7, width=60)
        self.output.pack(pady=10)

        # ===== Graphical Gantt Chart (Canvas) =====
        tk.Label(root, text="Gantt Chart", font=("Arial", 11, "bold")).pack()
        self.canvas_width = 800
        self.canvas_height = 250
        
        self.canvas_frame = tk.Frame(root)
        self.canvas_frame.pack(pady=10, padx=20, fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(self.canvas_frame, width=self.canvas_width, height=self.canvas_height, bg="white", relief=tk.SUNKEN, bd=2)
        self.canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

    def _validate_inputs(self):
        pid = self.pid_entry.get().strip()
        try:
            at = float(self.at_entry.get())
            bt = float(self.bt_entry.get())
        except ValueError:
            raise ValueError("Arrival/Burst Time must be numbers.")

        if pid == "": raise ValueError("Process ID is required.")
        if at < 0: raise ValueError("Arrival time must be >= 0.")
        if bt <= 0: raise ValueError("Burst time must be > 0.")
        return pid, at, bt

    def on_enter_add(self, *_):
        self.add_process()

    def add_process(self):
        try:
            pid, at, bt = self._validate_inputs()
            selected = self.tree.selection()

            if selected:
                item_id = selected[0]
                old_pid, _, _ = self.tree.item(item_id, "values")
                if any(existing_pid == pid and existing_pid != old_pid for existing_pid, _, _ in self.processes):
                    raise ValueError("Process ID must be unique.")
                for i, (existing_pid, _, _) in enumerate(self.processes):
                    if existing_pid == old_pid:
                        self.processes[i] = (pid, at, bt)
                        break
                self.tree.item(item_id, values=(pid, at, bt))
                self.tree.selection_remove(item_id)
            else:
                if any(existing_pid == pid for existing_pid, _, _ in self.processes):
                    raise ValueError("Process ID must be unique.")
                self.processes.append((pid, at, bt))
                self.tree.insert("", "end", values=(pid, at, bt))

            self.pid_entry.delete(0, tk.END)
            self.at_entry.delete(0, tk.END)
            self.bt_entry.delete(0, tk.END)
            self.delete_button.config(state=tk.DISABLED)
            self.pid_entry.focus_set()

        except ValueError as exc:
            messagebox.showerror("Error", str(exc) if str(exc) else "Invalid input!")

    def on_select_process(self, *_):
        selected = self.tree.selection()
        if not selected:
            self.delete_button.config(state=tk.DISABLED)
            return
        item_id = selected[0]
        pid, at, bt = self.tree.item(item_id, "values")
        self.pid_entry.delete(0, tk.END)
        self.pid_entry.insert(0, pid)
        self.at_entry.delete(0, tk.END)
        self.at_entry.insert(0, at)
        self.bt_entry.delete(0, tk.END)
        self.bt_entry.insert(0, bt)
        self.delete_button.config(state=tk.NORMAL)

    def delete_selected_process(self):
        selected = self.tree.selection()
        if not selected: return
        item_id = selected[0]
        pid, _, _ = self.tree.item(item_id, "values")
        self.processes = [proc for proc in self.processes if proc[0] != pid]
        self.tree.delete(item_id)
        self.clear_entries()

    def clear_entries(self):
        self.pid_entry.delete(0, tk.END)
        self.at_entry.delete(0, tk.END)
        self.bt_entry.delete(0, tk.END)
        self.delete_button.config(state=tk.DISABLED)
        self.pid_entry.focus_set()

    def clear_all(self):
        self.processes.clear()
        for item in self.tree.get_children():
            self.tree.delete(item)
        self.output.delete(1.0, tk.END)
        self.canvas.delete("all")
        self.clear_entries()

    def run_fcfs(self):
        if not self.processes:
            messagebox.showwarning("Warning", "No processes added!")
            return

        processes = sorted(self.processes, key=lambda x: (x[1], int(x[0]) if x[0].isdigit() else x[0]))

        n = len(processes)
        ct, tat, wt = [0] * n, [0] * n, [0] * n
        current_time = 0
        gantt = []

        for i in range(n):
            pid, at, bt = processes[i]

            if current_time < at:
                gantt.append(("Idle", current_time, at))
                current_time = at

            start_time = current_time
            current_time += bt

            ct[i] = current_time
            tat[i] = ct[i] - at
            wt[i] = tat[i] - bt

            gantt.append((pid, start_time, current_time))

        self.output.delete(1.0, tk.END)
        self.output.insert(tk.END, "PID\tAT\tBT\tCT\tTAT\tWT\n")
        self.output.insert(tk.END, "-"*50 + "\n")
        for i in range(n):
            self.output.insert(tk.END, f"{processes[i][0]}\t{processes[i][1]}\t{processes[i][2]}\t{ct[i]}\t{tat[i]}\t{wt[i]}\n")

        avg_tat = sum(tat) / n
        avg_wt = sum(wt) / n
        self.output.insert(tk.END, f"\nAvg TAT: {avg_tat:.2f}  |  Avg WT: {avg_wt:.2f}\n")

        self.draw_gantt_chart(gantt)

    def draw_gantt_chart(self, gantt):
        self.canvas.delete("all")
        if not gantt: return

        total_time = gantt[-1][2]
        if total_time == 0: return

        real_processes = [b for b in gantt if b[0] != "Idle"]
        unique_pids = []
        for b in real_processes:
            if b[0] not in unique_pids:
                unique_pids.append(b[0])

        padding_x = 40
        padding_y = 30
        usable_width = self.canvas_width - (padding_x * 2)
        scale = usable_width / total_time if total_time > 0 else 1
        
        row_height = 20
        row_spacing = 25

        needed_height = padding_y + len(unique_pids) * (row_height + row_spacing) + 40
        self.canvas.config(height=max(200, needed_height))

        self.canvas.create_line(padding_x, padding_y, padding_x + usable_width, padding_y, fill="black", width=2)
        
        t = 0.0
        step = 1 if total_time >= 5 else 0.5 # Tự điều chỉnh vạch chia cho đẹp
        while t <= total_time + 0.0001:
            x = padding_x + t * scale
            self.canvas.create_line(x, padding_y, x, needed_height - 20, fill="#E0E0E0", dash=(2, 2))
            self.canvas.create_line(x, padding_y - 5, x, padding_y, fill="black")
            self.canvas.create_text(x, padding_y - 15, text=f"{t:g}", font=("Arial", 8))
            t += step

        prev_x = None
        prev_y = None

        for block in gantt:
            pid, start, end = block
            
            if pid == "Idle":
                continue 

            row_idx = unique_pids.index(pid)
            
            x0 = padding_x + start * scale
            x1 = padding_x + end * scale
            y0 = padding_y + 15 + row_idx * (row_height + row_spacing)
            y1 = y0 + row_height
            mid_y = (y0 + y1) / 2

            if prev_x is not None and prev_y is not None:
                if prev_x == x0:
                    self.canvas.create_line(prev_x, prev_y, x0, mid_y, arrow=tk.LAST, dash=(4, 2), fill="#555555", width=1.5)
                else:
                    self.canvas.create_line(prev_x, prev_y, prev_x, mid_y, dash=(4, 2), fill="#555555", width=1.5)
                    self.canvas.create_line(prev_x, mid_y, x0, mid_y, arrow=tk.LAST, dash=(4, 2), fill="#555555", width=1.5)

            self.canvas.create_rectangle(x0, y0, x1, y1, fill="#8FAADC", outline="#2E528E", width=1.5)
            
            bar_width = x1 - x0
            if bar_width > 25:
                self.canvas.create_text((x0 + x1) / 2, mid_y, text=f"P{pid}", font=("Arial", 9, "bold"), fill="black")
            else:
                self.canvas.create_text(x1 + 15, mid_y, text=f"P{pid}", font=("Arial", 9, "bold"), fill="black")

            prev_x = x1
            prev_y = mid_y

In [3]:
ip = None
try:
    from IPython import get_ipython as _get_ipython
    ip = _get_ipython()
except Exception:
    pass

if ip is not None:
    try:
        ip.run_line_magic("gui", "tk")
    except Exception:
        pass

# Close previous window when rerunning the cell in notebook.
old_root = globals().get("root")
if old_root is not None:
    try:
        old_root.destroy()
    except Exception:
        pass

root = tk.Tk()
app = FCFSApp(root)

# In notebooks with %gui tk, the event loop is integrated, so no blocking mainloop is needed.
if ip is None:
    root.mainloop()